# 3 - Metadaten abholen

Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen. Dazu wird ein Unterordner {signature} im Ordner 'metadata' erstellt. 

### Metadaten aus Alma (SRU, marcxml)

Mit der MMS ID werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. 


### Datacite (OAI, XML)

Daten aus Zenodo-Repositories werden mit Datacite-Metadaten ins Archiv eingelagert.
Dazu wird die Zenodo-ID und das OAI-Set benötigt. (TODO)

### Weitere Metadaten

z.B. Dublin Core, OCR, etc. (TODO)

In [ ]:
import requests
import json
from datetime import datetime
from pathlib import Path
import config

# which metadata is available:
marc = config.marcxml
marc_url = config.marcxml_baseurl

datacite = config.datacite
datacite_url = config.datacite_baseurl
datacite_set = config.datacite_set

# general config:

urn = config.ingest_workflow
collection = config.collection_id
md_path = f'{collection}/{config.metadata_path}'
org_id = config.organisation_id

input_file = f"{collection}/{config.files_path}/{collection}_complete_set.json"

with open(input_file) as data_file:    
    data = json.load(data_file)
    for value in data:
        
        foldername = value["references"][-1]
        #print(foldername)
        identifiers = {}
        for item in value["identifiers"]:
            # split identifiers in dict
            [key, value] = item.split(':',1)
            identifiers[key] = value
        #print(identifiers)
        
        # create new directory for each signature (ignore, if it already exists)
        Path(f'{md_path}/{foldername}').mkdir(parents=True, exist_ok=True)
        
        # check for metadata settings:
        
        # marcxml data:
        
        if marc == 'true':
            
            # get mms_id
            mmsid = identifiers['mmsid']
            
            # get SRU response
            query = marc_url+mmsid
            response = requests.get(query)
            if response.status_code != 200:
                raise Exception(f"SRU request failed with status code {response.status_code}")

            # Save the response content as xml to a new directory
            marcxmlfile = f"{md_path}/{foldername}/{mmsid}.xml"

            with open(marcxmlfile, 'wb') as file:
                file.write(response.content)
                print(f"\nRecord with ID {mmsid} saved as {marcxmlfile}")
            
        
        # datacite and other metadata
        
        if datacite == 'true':
            
            # todo
            print("TODO")
            

        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))